# Silver - YouTube 2025

## Objetivo

Transformar os dados brutos de tendências do YouTube em uma base
padronizada para o Brasil no ano de 2025.

## Transformações

- seleção do arquivo histórico principal;
- filtro de registros do Brasil;
- padronização da data;
- recorte do ano de 2025;
- tipagem das métricas;
- padronização das categorias;
- identificação das categorias comparáveis com o CETIC.

## Categorias comparáveis

- Notícias
- Esportes
- Música
- Humor
- Games

In [0]:
from pyspark.sql import functions as F

In [0]:
caminho_youtube = (
    "/Volumes/workspace/mvp_bronze/raw_files/youtube_2025/"
    "youtube_trending_videos_global.parquet"
)

df_youtube = spark.read.parquet(caminho_youtube)

In [0]:
campos_necessarios = [
    "video_id",
    "video_trending_country",
    "video_trending__date",
    "video_category_id",
    "video_view_count",
    "video_like_count",
    "video_comment_count"
]

for campo in campos_necessarios:
    print(
        campo,
        "->",
        "OK" if campo in df_youtube.columns
        else "NÃO ENCONTRADO"
    )

In [0]:
df_youtube_br = (
    df_youtube
    .filter(
        F.trim(F.col("video_trending_country")) == "Brazil"
    )
)

In [0]:
df_youtube_br = (
    df_youtube_br
    .withColumn(
        "_data_texto_normalizada",
        F.regexp_replace(
            F.col("video_trending__date").cast("string"),
            r"\.",
            "-"
        )
    )
    .withColumn(
        "data_trending",
        F.expr(
            "try_cast(_data_texto_normalizada as date)"
        )
    )
)

In [0]:
df_youtube_2025 = (
    df_youtube_br
    .filter(
        F.year("data_trending") == 2025
    )
)

In [0]:
df_youtube_2025 = (
    df_youtube_2025
    .withColumn(
        "categoria_original",
        F.trim(
            F.col("video_category_id").cast("string")
        )
    )
    .withColumn(
        "views",
        F.expr("try_cast(video_view_count as bigint)")
    )
    .withColumn(
        "likes",
        F.expr("try_cast(video_like_count as bigint)")
    )
    .withColumn(
        "comments",
        F.expr("try_cast(video_comment_count as bigint)")
    )
)

In [0]:
display(
    df_youtube_2025
    .select(
        "video_id",
        "data_trending",
        "categoria_original",
        "views",
        "likes",
        "comments"
    )
    .limit(20)
)

In [0]:
display(
    df_youtube_2025
    .groupBy("categoria_original")
    .agg(
        F.count("*").alias("aparicoes_trending"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("aparicoes_trending"))
)

In [0]:
df_youtube_2025 = (
    df_youtube_2025
    .withColumn(
        "categoria",
        F.when(
            F.lower(F.col("categoria_original")) == "gaming",
            "Games"
        )
        .when(
            F.lower(F.col("categoria_original")) == "music",
            "Música"
        )
        .when(
            F.lower(F.col("categoria_original")) == "sports",
            "Esportes"
        )
        .when(
            F.lower(F.col("categoria_original")) == "news & politics",
            "Notícias"
        )
        .when(
            F.lower(F.col("categoria_original")) == "comedy",
            "Humor"
        )
        .when(
            F.lower(F.col("categoria_original")) == "film & animation",
            "Animações"
        )
        .when(
            F.lower(F.col("categoria_original")) == "education",
            "Tutoriais / Educação"
        )
        .when(
            F.lower(F.col("categoria_original")) == "people & blogs",
            "Influenciadores"
        )
        .when(
            F.lower(F.col("categoria_original")) == "howto & style",
            "Lifestyle / How-to"
        )
        .otherwise("Não comparável")
    )
)

In [0]:
df_youtube_2025 = (
    df_youtube_2025
    .withColumn(
        "nivel_comparabilidade",
        F.when(
            F.lower(F.col("categoria_original")).isin(
                "gaming",
                "music",
                "sports",
                "news & politics",
                "comedy",
                "film & animation"
            ),
            "Direta"
        )
        .when(
            F.lower(F.col("categoria_original")).isin(
                "education",
                "people & blogs"
            ),
            "Aproximada"
        )
        .when(
            F.lower(F.col("categoria_original")) == "howto & style",
            "Fraca"
        )
        .otherwise("Não comparável")
    )
    .withColumn(
        "categoria_comparavel",
        F.col("nivel_comparabilidade").isin(
            "Direta",
            "Aproximada"
        )
    )
)

In [0]:
display(
    df_youtube_2025
    .groupBy(
        "categoria_original",
        "categoria",
        "nivel_comparabilidade",
        "categoria_comparavel"
    )
    .agg(
        F.count("*").alias("aparicoes_trending"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("aparicoes_trending"))
)

In [0]:
display(
    df_youtube_2025
    .groupBy(
        "categoria_original",
        "categoria",
        "nivel_comparabilidade",
        "categoria_comparavel"
    )
    .agg(
        F.count("*").alias("aparicoes_trending"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("aparicoes_trending"))
)

In [0]:
display(
    df_youtube_2025
    .groupBy("nivel_comparabilidade")
    .agg(
        F.count("*").alias("aparicoes_trending"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("aparicoes_trending"))
)

In [0]:
display(
    df_youtube_2025
    .filter(
        F.col("nivel_comparabilidade").isin(
            "Direta",
            "Aproximada"
        )
    )
    .groupBy(
        "categoria",
        "nivel_comparabilidade"
    )
    .agg(
        F.count("*").alias("aparicoes_trending"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("videos_unicos"))
)

In [0]:
from pyspark.sql.window import Window

janela_video = Window.partitionBy("video_id")

df_youtube_2025 = (
    df_youtube_2025
    .withColumn(
        "primeira_data_trending",
        F.min("data_trending").over(janela_video)
    )
    .withColumn(
        "ultima_data_trending",
        F.max("data_trending").over(janela_video)
    )
)

In [0]:
df_youtube_2025 = (
    df_youtube_2025
    .withColumn(
        "dias_entre_primeira_ultima",
        F.datediff(
            F.col("ultima_data_trending"),
            F.col("primeira_data_trending")
        ) + 1
    )
)

In [0]:
[c for c in df_youtube_2025.columns if "title" in c.lower()]

In [0]:
df_youtube_silver = (
    df_youtube_2025
    .select(
        "video_id",
        "video_title",
        "data_trending",
        "primeira_data_trending",
        "ultima_data_trending",
        "dias_entre_primeira_ultima",
        "categoria_original",
        "categoria",
        "nivel_comparabilidade",
        "categoria_comparavel",
        "views",
        "likes",
        "comments"
    )
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "pais",
        F.lit("Brazil")
    )
    .withColumn(
        "fonte",
        F.lit("YouTube Trending Videos Dataset")
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
display(
    df_youtube_silver.limit(30)
)

In [0]:
display(
    df_youtube_silver
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(F.col("video_id").isNull(), 1).otherwise(0)
        ).alias("video_id_nulo"),

        F.sum(
            F.when(F.col("data_trending").isNull(), 1).otherwise(0)
        ).alias("data_nula"),

        F.sum(
            F.when(F.col("categoria_original").isNull(), 1).otherwise(0)
        ).alias("categoria_original_nula"),

        F.sum(
            F.when(F.col("views").isNull(), 1).otherwise(0)
        ).alias("views_nulo"),

        F.sum(
            F.when(F.col("likes").isNull(), 1).otherwise(0)
        ).alias("likes_nulo"),

        F.sum(
            F.when(F.col("comments").isNull(), 1).otherwise(0)
        ).alias("comments_nulo")
    )
)

In [0]:
display(
    df_youtube_silver
    .agg(
        F.min("data_trending").alias("data_minima"),
        F.max("data_trending").alias("data_maxima"),
        F.count("*").alias("registros"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
)

In [0]:
(
    df_youtube_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.mvp_silver.youtube_trending_br_2025"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_silver;

## Resultado da transformação Silver - YouTube

A fonte global de tendências do YouTube foi filtrada para o Brasil
e para o ano de 2025.

As categorias foram harmonizadas com a taxonomia do CETIC utilizando
três níveis de comparabilidade:

- Direta
- Aproximada
- Fraca

Correspondências diretas:
- Gaming → Games
- Music → Música
- Sports → Esportes
- News & Politics → Notícias
- Comedy → Humor
- Film & Animation → Animações

Correspondências aproximadas:
- Education → Tutoriais / Educação
- People & Blogs → Influenciadores

A categoria Howto & Style foi mantida como correspondência fraca
e não será utilizada no índice principal.

Categorias sem equivalência suficientemente segura foram preservadas
como `Não comparável`.

Tabela criada:

`workspace.mvp_silver.youtube_trending_br_2025`